# Installation

Session options:
 * Accelerator: GPU T4 x2
 * Language: Python
 * Persistence: File Only
 * Environment: Pin to original environment


In [ ]:
%%time
# update = False

import os
import stat
#!rm -rf /kaggle/working/venv
home_dir = '/kaggle/working'
python = '/kaggle/working/venv/bin/python'
pip = '/kaggle/working/venv/bin/pip'

def find_bin_folders(folder_path):
    bin_folders = []
    for root, dirs, files in os.walk(folder_path):
        for dir_name in dirs:
            if dir_name == 'bin':
                bin_folders.append(os.path.join(root, dir_name)) 
    return bin_folders

def installLibraries(home_dir, python, pip):
  %cd {home_dir}
  !{pip} install huggingface_hub xformers

  !{pip} install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128  #modify
  # !{pip} install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121  #moto
  # !{pip} install torchvision==0.23.0 --extra-index-url https://download.pytorch.org/whl/cu128  #customized pip,ok
    
  !{pip} install tensorflow[and-cuda]
  # TODO: download req.txt  
  !wget https://q4j3.c11.e2-5.dev/downloads/req.txt
  !{pip} install -r /kaggle/working/req.txt

  !pip install virtualenv

if not os.path.exists(f'{home_dir}/venv'):
    print('installing venv')
    os.chdir(home_dir)
    get_ipython().system(f'cd {home_dir}')
    
    get_ipython().system('virtualenv venv -p $(which python3.10)')
    installLibraries(home_dir, python, pip)
else:
    bin_folders = find_bin_folders('/kaggle/working/venv')
    if bin_folders:
      print("Found 'bin' folders:")
      for bin_folder in bin_folders:
        print(bin_folder)
        for filename in os.listdir(bin_folder):
            file_path = os.path.join(bin_folder, filename)
            if os.path.isfile(file_path):
                current_permissions = os.stat(file_path).st_mode
                 # Add execute permissions for the user, group, and others
                os.chmod(file_path, current_permissions | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

if not os.path.exists(f'{home_dir}/venv/bin/python3.10'):
    get_ipython().system('cp /usr/bin/python3.10 /kaggle/working/venv/bin/')
!ln -s /kaggle/working/venv/bin/python3.10 /kaggle/working/venv/bin/python
!ln -s /kaggle/working/venv/bin/python3.10 /kaggle/working/venv/bin/python3

%cd /kaggle/working
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd ComfyUI
# !git checkout 7fc3ccdcc2fb1f20c4b7dd4aca374db952fd66df

!{pip} install -r requirements.txt

!mkdir /tmp/models
!mkdir /tmp/models/checkpoints
!mkdir /tmp/models/clip
!mkdir /tmp/models/vae
!mkdir /tmp/models/unet
!mkdir /tmp/models/loras
!mkdir /tmp/models/diffusion_models
!mkdir /tmp/models/text_encoders

# Remove the following two lines to keep models in permanent storage
!rm -rf /kaggle/working/ComfyUI/models/checkpoints
!rm -rf /kaggle/working/ComfyUI/models/clip
!rm -rf /kaggle/working/ComfyUI/models/vae
!rm -rf /kaggle/working/ComfyUI/models/unet
!rm -rf /kaggle/working/ComfyUI/models/loras
!rm -rf /kaggle/working/ComfyUI/models/diffusion_models
!rm -rf /kaggle/working/ComfyUI/models/text_encoders
!rm -rf /kaggle/working/ComfyUI/models/text_encoders

!ln -s /tmp/models/checkpoints /kaggle/working/ComfyUI/models/checkpoints
!ln -s /tmp/models/clip /kaggle/working/ComfyUI/models/clip
!ln -s /tmp/models/vae /kaggle/working/ComfyUI/models/vae
!ln -s /tmp/models/unet /kaggle/working/ComfyUI/models/unet
!ln -s /tmp/models/loras /kaggle/working/ComfyUI/models/loras
!ln -s /tmp/models/diffusion_models /kaggle/working/ComfyUI/models/diffusion_models
!ln -s /tmp/models/text_encoders /kaggle/working/ComfyUI/models/text_encoders

checkpoints =  '/kaggle/working/ComfyUI/models/checkpoints'
link_path = checkpoints + '/temp-models'
temp_models = '/kaggle/temp/temp-models'

!mkdir /kaggle/temp
!mkdir $temp_models

if not os.path.exists(link_path):
    get_ipython().system(f'ln -s {temp_models} {checkpoints}')

# Install the node manager
update_manager = True
%cd /kaggle/working/ComfyUI/custom_nodes
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git
%cd ComfyUI-Manager
!git pull

# if update_manager:
#     get_ipython().system('git pull')
#!{pip} install -r requirements.txt

# Pinggy script
!wget https://raw.githubusercontent.com/wandaweb/jupyter-webui-tunneling/main/pinggy.py -O /kaggle/working/pinggy.py
    
# Second GPU offload  ComfyBootlegOffload(Force/Set CLIP Device,Force/Set VAE Device) does not work
# %cd /kaggle/working/ComfyUI/custom_nodes
# !wget https://gist.githubusercontent.com/city96/30743dfdfe129b331b5676a79c3a8a39/raw/ecb4f6f5202c20ea723186c93da308212ba04cfb/ComfyBootlegOffload.py
print('############################install completed############################')

--- 
# WebUI

## Start the WebUI with Pinggy
* Wait for the GUI to start.  
* Click the link that ends with .pinggy.link 😁
* If generation is still running after the link expires in an hour, wait for the generation to complete and restart the WebUI code block to get a new link

In [ ]:
# 导入 Kaggle Secrets 和 huggingface_hub
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, hf_hub_download

# 获取 Secrets 中的 HF_TOKEN
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HUGGINGFACE_TOKEN")
print(hf_token)

# 使用 HF_TOKEN 登录 Hugging Face
login(token=hf_token)

In [ ]:
############################################
########## install sd3.5 medium turbo and model #############
#url : https://huggingface.co/calcuis/sd3.5-medium-gguf
############################################
# 在comfyui manager下载ComfyUI-GGUF@1.1.0
# drag sd3.5_medium-q5_0.gguf (2.02GB) to > ./ComfyUI/models/unet
# drag clip_g.safetensors (1.39GB) to > ./ComfyUI/models/clip
# drag clip_l.safetensors (246MB) to > ./ComfyUI/models/clip
# drag t5xxl_fp8_e4m3fn.safetensors (4.89GB) to > ./ComfyUI/models/clip
# drag diffusion_pytorch_model.safetensors (168MB) to > ./ComfyUI/models/vae

# drag sd3.5_large_turbo-q4_0.gguf (4.77GB) to > ./ComfyUI/models/unet

repo_name_lst=[
    # 'calcuis/calcuis/sd3.5-large-gguf/sd3.5_large-q4_0.gguf',
    'calcuis/sd3.5-large-turbo/sd3.5_large_turbo-q4_0.gguf',
    # 'calcuis/sd3.5-medium-gguf/sd3.5_medium-q4_0.gguf',
    # 'calcuis/sd3.5-medium-gguf/sd3.5_medium-q5_0.gguf',
    # 'calcuis/sd3.5-medium-gguf/sd3.5_medium-q8_0.gguf',
    # 'calcuis/sd3.5-medium-gguf/sd3.5_medium-f16.gguf',
    'calcuis/sd3.5-medium-gguf/clip_g.safetensors',
    'calcuis/sd3.5-medium-gguf/clip_l.safetensors',
    'calcuis/sd3.5-medium-gguf/t5xxl_fp8_e4m3fn.safetensors',
    'calcuis/sd3.5-medium-gguf/diffusion_pytorch_model.safetensors',
  
]
repo_id_lst=[i.split('/')[0]+'/'+i.split('/')[1] for i in repo_name_lst]
filename_lst=[i.split('/')[2] for i in repo_name_lst]
local_dir_lst=['/unet/','/clip/','/clip/','/clip/','/vae/']    

print('repo_id_lst=',repo_id_lst)
print('filename_lst=',filename_lst)

In [ ]:
########## install wan2.2 #############
#url : https://huggingface.co/calcuis/wan2-gguf
# diffusion_models / wan2.2_ti2v_5B_fp16.safetensors
# text_encoders / umt5_xxl_fp8_e4m3fn_scaled.safetensors
# vae / wan2.2_vae.safetensors
repo_name_lst=[
    'QuantStack/Wan2.2-TI2V-5B-GGUF/Wan2.2-TI2V-5B-Q2_K.gguf',
    # 'QuantStack/Wan2.2-TI2V-5B-GGUF/Wan2.2-TI2V-5B-Q4_0.gguf',
    'calcuis/wan2-gguf/umt5xxl_fp8_e4m3fn_scaled.safetensors',
    'QuantStack/Wan2.2-TI2V-5B-GGUF/VAE/Wan2.2_VAE.safetensors',
  
]
repo_id_lst=[i.split('/')[0]+'/'+i.split('/')[1] for i in repo_name_lst]
filename_lst=[i.split('/')[-1] for i in repo_name_lst]
filename_lst[2]='VAE/'+filename_lst[2]
local_dir_lst=[
    # '/diffusion_models/',
    '/unet/',
    '/text_encoders/',
    '/vae/'
]

print('repo_id_lst=',repo_id_lst)
print('filename_lst=',filename_lst)

In [ ]:
############################################
##########install nunchaku#############
############################################
# !/kaggle/working/venv/bin/pip install huggingface_hub xformers
# !/kaggle/working/venv/bin/pip install torchvision==0.23.0 --extra-index-url https://download.pytorch.org/whl/cu128
# 如果报错ComfyUI-nunchaku 1.0.0 is not compatible with nunchaku 0.3.2. Please update nunchaku to a supported version in ['v1.0.0'].v1.0.0 currently is a nightly version. 
# 重新在comfyui manager下载nunchaku插，版本可选0.34
!/kaggle/working/venv/bin/python -m pip install https://github.com/nunchaku-tech/nunchaku/releases/download/v0.3.2/nunchaku-0.3.2+torch2.8-cp310-cp310-linux_x86_64.whl
# then install ComfyUI-nunchaku in comfyui manager

In [ ]:
############################################
########## install nunchaku models#############
############################################
# 下载模型文件
# 替换 repo_id 和 filename 为你需要的模型和文件
repo_id_lst=[
    'black-forest-labs/FLUX.1-schnell',
    'comfyanonymous/flux_text_encoders',
    'nunchaku-tech/nunchaku-flux.1-schnell',
    'comfyanonymous/flux_text_encoders'
]
filename_lst=[
    'ae.safetensors',
    't5xxl_fp16.safetensors',
    'svdq-int4_r32-flux.1-schnell.safetensors',
    'clip_l.safetensors'
]
# local_dir_lst=['/vae/','text_encoders/','checkpoints/','clip/']
local_dir_lst=['/vae/','text_encoders/','unet/','clip/']    
# ! ll /tmp/models/    

In [ ]:
############################################
########## download models
############################################

def download_model(repo_id_lst,filename_lst,local_dir_lst,i):
    model_path = hf_hub_download(
        repo_id=repo_id_lst[i],  # 替换为你的模型 ID
        filename=filename_lst[i],                    # 替换为具体的模型文件
        local_dir="/tmp/models/"+local_dir_lst[i]  # 指定保存路径
    )
    
    print(f"模型已下载到: {model_path}")    
# download_model(repo_id_lst,filename_lst,local_dir_lst,2)
for i in range(len(repo_id_lst)):
    pass
    download_model(repo_id_lst,filename_lst,local_dir_lst,i)

In [ ]:
# Starting the Web UI with pinggy

%cd /kaggle/working/ComfyUI
!python /kaggle/working/pinggy.py --command='/kaggle/working/venv/bin/python /kaggle/working/ComfyUI/main.py' --port=8188
# !python /kaggle/working/pinggy.py --command='/kaggle/working/venv/bin/python /kaggle/working/ComfyUI/main.py --lowvram' --port=8188

---
# Model Management

## Install a model

Copy the model URL to the model_url field. Make sure the model can be accessed publicly, without being signed into a website.

In [ ]:
#### Install a model in permanent storage
# Make sure Persistence is set to "Files only" or "Variables and Files"
model_url = 'https://civitai.com/api/download/models/782002'
model_name = 'JuggernautXL.safetensors'

%cd $checkpoints
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a LoRA in permanent storage
model_url = 'https://civitai.com/api/download/models/137124?type=Model&format=SafeTensor'
model_name = 'DreamArt.safetensors'

%cd /kaggle/working/ComfyUI/models/loras
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a model in temporary storage
#model_url = 'https://civitai.com/api/download/models/160191?type=Model&format=SafeTensor&size=full&fp=fp16'
#model_name = 'YamersRealism.safetensors'
# model_url = 'https://civitai.com/api/download/models/456751'
# model_name = 'HelloWorld-XL.safetensors' 
# https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/diffusion_models/wan2.2_ti2v_5B_fp16.safetensors
# https://huggingface.co/Comfy-Org/Wan_2.1_ComfyUI_repackaged/resolve/main/split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors
model_url = 'https://huggingface.co/Comfy-Org/Wan_2.2_ComfyUI_Repackaged/resolve/main/split_files/vae/wan2.2_vae.safetensors'
model_name = 'wan2.2_vae.safetensors' 
# temp_models = '/kaggle/temp/temp-models'
%cd $temp_models
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

In [ ]:
# Install a model in temporary storage2  for workflow : lora
# loras/ MoXinV1.safetensors
model_url ='https://civitai.com/api/download/models/14856?type=Model&format=SafeTensor&size=full&fp=fp16'
model_name='loras/blindbox_v1_mix.safetensors'
model_url ='https://civitai.com/api/download/models/32988?type=Model&format=SafeTensor&size=full&fp=fp16'
model_name='loras/MoXinV1.safetensors'
model_url ='https://civitai.com/api/download/models/128713?type=Model&format=SafeTensor&size=pruned&fp=fp16'
model_name='checkpoints/dreamshaper_8.safetensors'
# temp_models = '/temp/temp-models'
%cd /tmp/models
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

## Download a model for a custom node

In [ ]:
model_folder = '/kaggle/working/ComfyUI/custom_nodes/my_node/models'
model_url = ''
model_name = 'model.safetensors'

%cd $model_folder
# /kaggle/working/ComfyUI/custom_nodes/my_node/models
# /kaggle/working/ComfyUI/custom_nodes
# !/kaggle/working/venv/bin/pip install 
get_ipython().system(f'wget -O "{model_name}" "{model_url}"')

---
# File Browser

## Install FileBrowser

In [ ]:
%cd /kaggle
!wget https://github.com/filebrowser/filebrowser/releases/download/v2.27.0/linux-amd64-filebrowser.tar.gz
!tar xvfz linux-amd64-filebrowser.tar.gz
!chmod a+x /kaggle/filebrowser
!/kaggle/filebrowser config init 
!/kaggle/filebrowser config set --auth.method=noauth > /dev/null
!/kaggle/filebrowser config set --branding.theme=dark > /dev/null
!/kaggle/filebrowser users add admin admin 
!/kaggle/filebrowser config export "/kaggle/config.json"

## Run FileBrowser

In [ ]:
%cd /kaggle
!chmod a+x /kaggle/filebrowser

!python /kaggle/working/pinggy.py --command='/kaggle/filebrowser -c "/kaggle/working/config.json"' --port=8080

# 
# Delete a model

In [ ]:
# List permanent models
!ls -la $checkpoints

# Delete a model
model_to_delete = '/kaggle/working/ComfyUI/models/checkpoints/model.safetensors'
!rm $model_to_delete

In [ ]:
# Check the size of a model
!du -sh /kaggle/working/ComfyUI/models/loras/harrlogos.safetensors

# 
# Delete everything in the working folder

In [ ]:
# Delete the working folder
!rm -rf /kaggle/working/*